In [ ]:
# unzip file into another location
# !unzip "/content/drive/MyDrive/Teaching/PGA59/dl/cnn_small/intel.zip" -d "/content/drive/MyDrive/Teaching/PGA59/dl/cnn_small/intel"

In [ ]:
import tensorflow as tf
from tensorflow.keras import models
from tensorflow.keras import layers
# image data read
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True,
    shear_range=0.1
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    "/content/drive/MyDrive/Teaching/PGAPRO02/dl/chest_xray/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'   # or 'categorical' if multi-class
)

val_gen = val_datagen.flow_from_directory(
    "/content/drive/MyDrive/Teaching/PGAPRO02/dl/chest_xray/test",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')  # binary
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
from tensorflow.keras.metrics import Recall

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', Recall()]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=2)
]

In [ ]:
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)

163.0

In [ ]:
model.fit(train_gen, validation_data=val_gen, epochs=3)

Epoch 1/3
 36/163 ━━━━━━━━━━━━━━━━━━━━ 25:35 12s/step - Recall: 0.8326 - accuracy: 0.6457 - loss: 0.8645

KeyboardInterrupt: 

In [ ]:
model.save("cnn_model.h5")


In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("cnn_model")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_image(model, img_path):
    img = image.load_img(img_path, target_size=(224,224), color_mode='grayscale')

    img_array = image.img_to_array(img)   # shape: (224,224,1)
    img_array = img_array / 255.0

    img_array = np.expand_dims(img_array, axis=0)  # (1,224,224,1)

    pred = model.predict(img_array)[0][0]
    p = model.predict(img_array)

    label = "Pneumonia" if pred > 0.5 else "Normal"
    confidence = pred if pred > 0.5 else 1 - pred

    return label, float(confidence), p

In [ ]:
model.evaluate(val_gen)

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 248ms/step - Recall: 0.9897 - accuracy: 0.8141 - loss: 0.5423


[0.5422766208648682, 0.8141025900840759, 0.9897435903549194]

In [ ]:
predict_image(model, "/content/NORMAL2-IM-1431-0001.jpeg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


('Normal', 0.5669307708740234, array([[0.43306923]], dtype=float32))

In [ ]:
# New code

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =========================
# 1. Paths
# =========================
train_dir = "/content/drive/MyDrive/Teaching/PGAPRO02/dl/chest_xray/train"
val_dir = "/content/drive/MyDrive/Teaching/PGAPRO02/dl/chest_xray/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# =========================
# 2. Data Generators (Grayscale)
# =========================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# =========================
# 3. Model Architecture
# =========================
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

# =========================
# 4. Compile
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# 5. Callbacks
# =========================
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2)
]

# =========================
# 6. Train
# =========================
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks
)


Found 5216 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 767s 5s/step - accuracy: 0.8961 - loss: 0.3969 - val_accuracy: 0.6250 - val_loss: 7.9021 - learning_rate: 1.0000e-04
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 83s 509ms/step - accuracy: 0.9270 - loss: 0.1871 - val_accuracy: 0.6250 - val_loss: 9.2332 - learning_rate: 1.0000e-04
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 83s 510ms/step - accuracy: 0.9291 - loss: 0.1845 - val_accuracy: 0.6266 - val_loss: 3.3905 - learning_rate: 1.0000e-04
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 82s 505ms/step - accuracy: 0.9360 - loss: 0.1691 - val_accuracy: 0.7035 - val_loss: 1.1620 - learning_rate: 1.0000e-04
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 83s 506ms/step - accuracy: 0.9381 - loss: 0.1642 - val_accuracy: 0.8285 - val_loss: 0.6290 - learning_rate: 1.0000e-04
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 83s 510ms/step - accuracy: 0.9438 - loss: 0.1474 - val_accuracy: 0.8894 - val_loss: 0.3786 - learning_rate: 1.0000e-04
Epoch 7/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 84s 514m

In [ ]:
# =========================
# 7. Save Model + Class Map
# =========================

model.save("/content/drive/MyDrive/Teaching/PGAPRO02/dl/cnn_grayscale_model.h5")

import json
with open("/content/drive/MyDrive/Teaching/PGAPRO02/dl/class_map.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print("Model saved successfully")

Model saved successfully


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import json

# =========================
# 1. Load Model
# =========================
model = tf.keras.models.load_model("/content/drive/MyDrive/Teaching/PGAPRO02/dl/cnn_grayscale_model.h5")

# =========================
# 2. Load Class Map
# =========================
with open("/content/drive/MyDrive/Teaching/PGAPRO02/dl/class_map.json") as f:
    class_map = json.load(f)

# reverse mapping
class_map = {v:k for k,v in class_map.items()}

# =========================
# 3. Prediction Function
# =========================
def predict_image(img_path):
    img = image.load_img(
        img_path,
        target_size=(224,224),
        color_mode='grayscale'
    )

    img_array = image.img_to_array(img) / 255.0

    # shape: (224,224,1) → (1,224,224,1)
    img_array = np.expand_dims(img_array, axis=0)

    pred = model.predict(img_array)[0][0]

    class_id = 1 if pred > 0.5 else 0
    label = class_map[class_id]

    return {
        "probability": float(pred),
        "class": label
    }
# =========================
# 4. Test Prediction
# =========================
result = predict_image("/content/person1_bacteria_1.jpeg")
print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 917ms/step
{'probability': 0.9997389912605286, 'class': 'PNEUMONIA'}


In [ ]:
result = predict_image("/content/IM-0127-0001.jpeg")
print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
{'probability': 0.0002165434416383505, 'class': 'NORMAL'}
